# Google Sheets Export
#### Pushing remote salary benchmark findings to google sheets from sql
**Input:** Query Results fromt SQL analysis  
**Author: Kunal Verma**

In [8]:
import numpy as np
import pandas as pd
import sqlite3
import gspread
from gspread_dataframe import set_with_dataframe
from google.oauth2.service_account import Credentials

print("libraries have been loaded")

libraries have been loaded


In [9]:
# connecting to google sheets
scope = ['https://www.googleapis.com/auth/spreadsheets', 'https://www.googleapis.com/auth/drive']

creds = Credentials.from_service_account_file('../credentials.json', scopes=scope)
client = gspread.authorize(creds)

# open the sheet
sheet = client.open('remote salary benchmark')
print("connected to google sheets.")

connected to google sheets.


In [10]:
# reconnect to db to get all query results
conn = sqlite3.connect('../data/salary_benchmark.db')

q1 = pd.read_sql_query("""
    SELECT employee_residence as country,
        ROUND(AVG(salary_in_usd), 0) as avg_salary,
        ROUND(MIN(salary_in_usd), 0) as min_salary,
        ROUND(MAX(salary_in_usd), 0) as max_salary,
        COUNT(*) as total_records
    FROM levels
    GROUP BY employee_residence
    ORDER BY avg_salary DESC
""", conn)

q2 = pd.read_sql_query("""
    SELECT employee_residence as country,
        experience_level,
        ROUND(AVG(salary_in_usd), 0) as avg_salary,
        COUNT(*) as total_records
    FROM levels
    GROUP BY employee_residence, experience_level
    ORDER BY country, avg_salary DESC
""", conn)

q4 = pd.read_sql_query("""
    SELECT us.job_title,
        ROUND(us.avg_salary, 0) as us_avg,
        ROUND(ind.avg_salary, 0) as india_avg,
        ROUND((us.avg_salary - ind.avg_salary) / us.avg_salary * 100, 1) as gap_pct
    FROM 
        (SELECT job_title, AVG(salary_in_usd) as avg_salary 
         FROM levels WHERE employee_residence = 'US' 
         GROUP BY job_title HAVING COUNT(*) > 2) us
    JOIN 
        (SELECT job_title, AVG(salary_in_usd) as avg_salary 
         FROM levels WHERE employee_residence = 'IN' 
         GROUP BY job_title HAVING COUNT(*) > 1) ind
    ON us.job_title = ind.job_title
    ORDER BY gap_pct DESC
    LIMIT 10
""", conn)

conn.close()
print("queries done ✓")

queries done ✓


In [11]:
# push to google sheets — one tab per query

# tab 1 — median salary by country
ws1 = sheet.add_worksheet(title='salary by country', rows=100, cols=20)
set_with_dataframe(ws1, q1)

# tab 2 — salary by experience level
ws2 = sheet.add_worksheet(title='salary by experience', rows=100, cols=20)
set_with_dataframe(ws2, q2)

# tab 3 — india vs us gap
ws3 = sheet.add_worksheet(title='india vs us gap', rows=100, cols=20)
set_with_dataframe(ws3, q4)

print("all data pushed to google sheets ✓")

all data pushed to google sheets ✓
